# 5-minute tour of `pypwsqc`

This is a very brief overview of features available in the `pypwsqc` package for quality control of Personal Weather Station (PWS) data. A more comprehensive overview can be found in the `pypwsqc` [documentation](https://pypwsqc.readthedocs.io).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import poligrain as plg
import xarray as xr

import pypwsqc.indicator_correlation as ic

warnings.simplefilter(action="ignore", category=FutureWarning)

## Load example data

We use an open PWS dataset from Amsterdam published by de Vos et al. (2019). The cell below downloads a NetCDF file (~5 MB) to the current directory. We also load a reference dataset (gauge-adjusted radar) used by the indicator correlation filter.

In [ ]:
!curl -OL https://github.com/OpenSenseAction/OS_data_format_conventions/raw/main/notebooks/data/OpenSense_PWS_example_format_data.nc

In [ ]:
ds_pws = xr.open_dataset("OpenSense_PWS_example_format_data.nc")
ds_pws.load()

# Project coordinates to a metric system (EPSG:25832 for the Netherlands)
ds_pws.coords["x"], ds_pws.coords["y"] = plg.spatial.project_point_coordinates(
    ds_pws.longitude,
    ds_pws.latitude,
    target_projection="EPSG:25832",
)
ds_pws

As reference data for the indicator correlation filter (so-called primary stations, cf. Bárdossy et al. (2021)), we use time series from pixels of the gauge-adjusted KNMI radar product over the Amsterdam metropolitan area.

In [ ]:
!curl -OL https://github.com/OpenSenseAction/pypwsqc/raw/main/docs/notebooks/data/RadarRef_AMS.nc

ds_ref = xr.open_dataset("RadarRef_AMS.nc")
ds_ref.load()
ds_ref.coords["x"], ds_ref.coords["y"] = plg.spatial.project_point_coordinates(
    ds_ref.lon,
    ds_ref.lat,
    target_projection="EPSG:25832",
)
ds_ref

## Indicator correlation filter

The indicator correlation filter detects PWS with data quality issues by comparing the spatial correlation structure of the PWS data with that of the reference data. First, we calculate the distance and indicator correlation matrices for the reference data, which is assumed to represent the correct spatial pattern of precipitation.

In [ ]:
# Distance and indicator correlations of reference data
dist_mtx_ref, indcorr_mtx_ref = ic.indicator_distance_matrix(
    ds_ref.rainfall,
    ds_ref.rainfall,
    max_distance=30e3,
    prob=0.99,
    min_valid_overlap=2 * 24 * 30,
)

In [ ]:
plt.scatter(dist_mtx_ref, indcorr_mtx_ref, color="red", s=10, label="Ref")
plt.ylim(0, 1)
plt.xlim(0, 30e3)
plt.ylabel("Indicator Correlation [-]")
plt.xlabel("Distance [m]")
plt.title("Indicator Correlation over Distance for Reference Data")
plt.legend();

Next, we calculate the distance and indicator correlation matrices between the PWS and the reference data.

In [ ]:
dist_mtx_ref_pws, indcorr_mtx_ref_pws = ic.indicator_distance_matrix(
    da_a=ds_ref.rainfall,
    da_b=ds_pws.rainfall,
    prob=0.99,
    min_valid_overlap=2 * 24 * 30,
)

Now we apply the indicator correlation filter. It flags PWS whose correlation structure deviates from the reference, indicating data quality issues.

In [ ]:
indcorr_results = ic.ic_filter(
    indicator_correlation_matrix_ref=indcorr_mtx_ref,
    distance_correlation_matrix_ref=dist_mtx_ref,
    indicator_correlation_matrix=indcorr_mtx_ref_pws,
    distance_matrix=dist_mtx_ref_pws,
    max_distance=20000,
    bin_size=1000,
    quantile_bin_ref=0.1,
    quantile_bin_pws=0.5,
    threshold=0.05,
)
indcorr_results

## Analysis of the results

The filter returns an `xarray.Dataset` with the indicator correlation matrix, the distance matrix, a boolean flag `indcorr_good` indicating whether each PWS was accepted, and an `indcorr_score` metric. Let's see how many PWS were accepted.

In [ ]:
print(
    str(indcorr_results.indcorr_good.data.sum())
    + " of "
    + str(len(indcorr_results.id_neighbor))
    + " PWS were accepted"
)

In [ ]:
indcorr_results.plot.scatter(x="dist", y="indcorr", color="grey", alpha=0.5, s=10)
plt.scatter(dist_mtx_ref, indcorr_mtx_ref, color="r", alpha=0.5, s=10, label="Ref")
plt.ylim(0, 1)
plt.xlim(0, 30e3)
plt.ylabel("Indicator Correlation [-]")
plt.xlabel("Distance [m]")
plt.legend();

## Summary

That's the core PWS quality control workflow in `pypwsqc`: load PWS and reference data → calculate distance and indicator correlation matrices → apply the indicator correlation filter to flag PWS with data quality issues. Explore the [documentation](https://pypwsqc.readthedocs.io) for more filters (e.g. peak removal, bias correction).